In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/psfc.npy
/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/t2.npy
/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/SO2.npy
/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/NMVOC_finn.npy
/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/bio.npy
/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/rain.npy
/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/u10.npy
/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in/swdown.npy
/kaggle/input/competitions/anrf-aise-hack-pha

In [2]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from scipy.io import loadmat

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

Device: cuda


In [3]:
RAW_PATH = "/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/raw"
TEST_PATH = "/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in"
STATS_PATH = "/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/stats/feat_min_max.mat"

MONTHS = ["APRIL_16", "JULY_16", "OCT_16", "DEC_16"]

FEATURES = [
    "PM25",
    "u10", "v10",
    "pblh",
    "t2",
    "NOx", "SO2", "NH3",
    "rain", "q2"
]

TARGET = "cpm25"

INPUT_STEPS = 10
OUTPUT_STEPS = 16
STRIDE = 2

BATCH_SIZE = 2
EPOCHS = 5
LR = 1e-3

In [4]:
min_max = loadmat(STATS_PATH)

min_vals = {f: min_max[f"{f}_min"].item() for f in FEATURES}
max_vals = {f: min_max[f"{f}_max"].item() for f in FEATURES}

EMISSION = ["NH3", "NOx", "SO2"]

In [5]:
class AirQualityDataset(Dataset):
    def __init__(self, mode="train", val_split=0.2, seed=42):
        self.samples = []

        def normalize(arr, feat):
            arr = (arr - min_vals[feat]) / (max_vals[feat] - min_vals[feat])

            if feat in ["u10", "v10"]:
                arr = 2 * arr - 1

            if feat in EMISSION:
                arr = np.clip(arr, 0, 1)

            return arr.astype(np.float32)

        for month in MONTHS:
            path = os.path.join(RAW_PATH, month)

            data = {}
            for f in FEATURES + [TARGET]:
                arr = np.load(os.path.join(path, f"{f}.npy"))
                data[f] = normalize(arr, f) if f != TARGET else arr.astype(np.float32)

            T = data[TARGET].shape[0]

            for i in range(0, T - (INPUT_STEPS + OUTPUT_STEPS), STRIDE):
                x = []
                for f in FEATURES:
                    x.append(data[f][i:i+INPUT_STEPS])

                x = np.stack(x, axis=2)
                y = data[TARGET][i+INPUT_STEPS:i+INPUT_STEPS+OUTPUT_STEPS]

                self.samples.append((x, y))

        threshold = np.percentile([s[1].mean() for s in self.samples], 80)

        high_samples = [s for s in self.samples if s[1].mean() > threshold]
        self.samples += high_samples * 2

        np.random.seed(seed)
        idx = np.random.permutation(len(self.samples))
        split = int((1 - val_split) * len(idx))

        if mode == "train":
            idx = idx[:split]
        else:
            idx = idx[split:]

        self.samples = [self.samples[i] for i in idx]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        x, y = self.samples[idx]
        return torch.tensor(x), torch.tensor(y)

In [6]:
class TestDataset(Dataset):
    def __init__(self):
        self.data = {}

        for f in FEATURES:
            self.data[f] = np.load(os.path.join(TEST_PATH, f"{f}.npy"))

        self.N = self.data[FEATURES[0]].shape[0]

    def normalize(self, arr, feat):
        arr = (arr - min_vals[feat]) / (max_vals[feat] - min_vals[feat])
        return arr.astype(np.float32)

    def __len__(self):
        return self.N

    def __getitem__(self, idx):
        x = []
        for f in FEATURES:
            arr = self.normalize(self.data[f][idx], f)
            x.append(arr)

        x = np.stack(x, axis=1)
        return torch.tensor(x)

In [7]:
train_dataset = AirQualityDataset(mode="train")
val_dataset = AirQualityDataset(mode="val")
test_dataset = TestDataset()

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=1)

print(len(train_dataset), len(val_dataset))

1585 397


In [8]:
class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c, k=3, s=1, p=1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_c, out_c, k, s, p),
            nn.BatchNorm2d(out_c),
            nn.GELU()
        )

    def forward(self, x):
        return self.net(x)

In [24]:
class Encoder(nn.Module):
    def __init__(self, in_c, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            ConvBlock(in_c, 32),
            ConvBlock(32, 64, s=2),
            ConvBlock(64, hidden, s=2)
        )

    def forward(self, x):
        return self.net(x)


class Decoder(nn.Module):
    def __init__(self, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(hidden, 64, 4, 2, 1),
            nn.GELU(),
            nn.ConvTranspose2d(64, 32, 4, 2, 1),
            nn.GELU(),
            nn.Conv2d(32, 1, 3, padding=1)
        )

    def forward(self, x):
        return self.net(x)

In [25]:
class TemporalAttention(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.attn = nn.MultiheadAttention(dim, num_heads=4, batch_first=True)
        self.norm = nn.LayerNorm(dim)

    def forward(self, x):
        B, T, C, H, W = x.shape

        x = x.reshape(B, T, C*H*W)
        out, _ = self.attn(x, x, x)

        x = self.norm(x + out)

        return x.reshape(B, T, C, H, W)


class SimVP(nn.Module):
    def __init__(self, C):
        super().__init__()

        # 🔥 Feature gating
        self.feature_mlp = nn.Sequential(
            nn.Linear(C, C//4),
            nn.ReLU(),
            nn.Linear(C//4, C),
            nn.Sigmoid()
        )
        self.encoder = Encoder(C)
        self.decoder = Decoder()
        self.temporal = None

        self.project = nn.Linear(INPUT_STEPS, OUTPUT_STEPS)

    def forward(self, x):
        feat = feat.reshape(B*OUTPUT_STEPS, C2, H2, W2)
        out = self.decoder(feat)
        x_input = x.clone()
        B, T, C, H, W = x.shape

        # Apply feature gating
        weights = self.feature_mlp(x.mean(dim=(1,3,4)))
        weights = weights.unsqueeze(1).unsqueeze(-1).unsqueeze(-1)
        
        x = x * weights

        x = x.reshape(B*T, C, H, W)
        feat = self.encoder(x)

        _, C2, H2, W2 = feat.shape

        if self.temporal is None:
            self.temporal = TemporalAttention(C2 * H2 * W2).to(feat.device)
            
        feat = feat.reshape(B, T, C2, H2, W2)

        feat = self.temporal(feat)

        feat = feat.permute(0, 2, 3, 4, 1).contiguous()
        feat = self.project(feat)
        feat = feat.permute(0, 4, 1, 2, 3)

        feat = feat.reshape(B*OUTPUT_STEPS, C2, H2, W2)
        out = self.decoder(feat)

        out = out.reshape(B, OUTPUT_STEPS, H, W)

        last_frame = x_input[:, -1, 0]
        baseline = last_frame.unsqueeze(1).repeat(1, OUTPUT_STEPS, 1, 1)
        return out + baseline

In [22]:
def smape(pred, target):
    return torch.mean(torch.abs(pred - target) / (torch.abs(pred) + torch.abs(target) + 1e-5))

def loss_fn(pred, target):
    mse = (pred - target) ** 2

    weight = 1 + 2 * (target > 0.6).float()

    smooth = torch.mean((pred[:,1:] - pred[:,:-1])**2)

    return (weight * mse).mean() + 0.1 * smooth

In [23]:
model = SimVP(C=140).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
best_val = float("inf")

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0

    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        if np.random.rand() < 0.3:
            B, T, C, H, W = x.shape
        
            x = x.cpu()
            y = y.cpu()
        
            x = x.reshape(B*T, C, H, W)
            x = F.interpolate(x, scale_factor=0.5, mode='bilinear')
            _, C, H2, W2 = x.shape
            x = x.reshape(B, T, C, H2, W2)
        
            By, Ty, Hy, Wy = y.shape
            y = y.reshape(By*Ty, 1, Hy, Wy)
            y = F.interpolate(y, scale_factor=0.5, mode='bilinear')
            _, _, H2, W2 = y.shape
            y = y.reshape(By, Ty, H2, W2)
        
            x = x.to(DEVICE)
            y = y.to(DEVICE)
        pred = model(x)
        loss = loss_fn(pred, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    model.eval()
    val_loss = 0

    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)

            pred = model(x)
            val_loss += loss_fn(pred, y).item()

    print(f"Epoch {epoch} | Train {train_loss:.3f} | Val {val_loss:.3f}")
    scheduler.step()
    if val_loss < best_val:
        best_val = val_loss
        torch.save(model.state_dict(), "/kaggle/working/best_model.pth")
        print("Saved best model")

UnboundLocalError: cannot access local variable 'feat' where it is not associated with a value

In [ ]:
model.load_state_dict(torch.load("/kaggle/working/best_model.pth"))
model.eval()

In [ ]:
all_preds = []

for flip in [False, True]:
    preds = []

    with torch.no_grad():
        for x in test_loader:
            x = x.to(DEVICE)

            if flip:
                x = torch.flip(x, dims=[-1])

            pred = model(x)

            if flip:
                pred = torch.flip(pred, dims=[-1])

            preds.append(pred.cpu().numpy())

    preds = np.concatenate(preds, axis=0)
    all_preds.append(preds)

preds = np.mean(all_preds, axis=0)


print("Before reshape:", preds.shape)

preds = np.transpose(preds, (0, 2, 3, 1))

print("Final shape:", preds.shape)

np.save("/kaggle/working/preds.npy", preds)